In [ ]:
# Load environment and enable LangSmith tracing
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
)

In [ ]:
# Configure DeepEval for OpenAI evaluation and optional Confident AI publishing
import os
from pathlib import Path
from dotenv import load_dotenv

# Put CONFIDENT_API_KEY in the notebook-local .env to publish results to the dashboard.
load_dotenv(Path.cwd() / ".env", override=True)
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "600")
os.environ["DEEPEVAL_DISABLE_DOTENV"] = "1"

import deepeval
from deepeval.confident.api import get_confident_api_key
from deepeval.models import OpenAIModel

print(f"DeepEval version: {__import__('importlib.metadata').metadata.version('deepeval')}")
print(f"Confident AI key configured: {get_confident_api_key() is not None}")

judge = OpenAIModel(
    model="gpt-4o-mini",
    temperature=0.0,
)


In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import OpenAIModel

if "judge" not in globals():
    judge = OpenAIModel(
        model="gpt-4o-mini",
        temperature=0.0,
    )

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)

score = answer_relevancy_metric.measure(test_case)
print(score)
print(answer_relevancy_metric)


In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric
from deepeval.models import OpenAIModel

judge = OpenAIModel(
    model="gpt-4o-mini",
    temperature=0.0,
    
)

context_precision_metric = ContextualPrecisionMetric(model=judge)

test_case = LLMTestCase(
    input="What is the capital of India?",
    actual_output="New Delhi",
    expected_output="New Delhi is the capital of India.",
    retrieval_context=["New Delhi is the capital city of India."]
)

score = context_precision_metric.measure(test_case)

print("Score:", context_precision_metric.score)
print("Success:", context_precision_metric.success)
print("Breakdown:", context_precision_metric.score_breakdown)

In [ ]:
# Run the evaluation and publish the result to Confident AI when CONFIDENT_API_KEY is configured
import os

os.environ.setdefault("DEEPEVAL_RESULTS_FOLDER", "./deepeval-results")

from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import OpenAIModel
from deepeval.evaluate import evaluate, CacheConfig

judge = OpenAIModel(
    model="gpt-4o-mini",
    temperature=0.0,
)

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)


evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)


In [ ]:
# Connect DeepEval to Confident AI
# Put your key in the project-root .env as: CONFIDENT_API_KEY=confident_xxx

import os

from dotenv import load_dotenv

# override=True so edits to .env take effect without restarting the kernel
load_dotenv(override=True)

# qwen3:8b is a reasoning model and can exceed DeepEval's default 88.5s per-attempt timeout
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "600")

import deepeval
from deepeval.models import OllamaModel

api_key = os.getenv("CONFIDENT_API_KEY")
if not api_key or "paste_your_key" in api_key:
    raise ValueError(
        "Set CONFIDENT_API_KEY in the project-root .env to your real Confident AI key."
    )
deepeval.login(api_key=api_key)

judge = OllamaModel(
    model="qwen3:8b",
    base_url="http://localhost:11434",
    temperature=0.0,
)

In [ ]:
import os

os.environ.setdefault("DEEPEVAL_RESULTS_FOLDER", "./deepeval-results")

from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import OpenAIModel
from deepeval.evaluate import evaluate, CacheConfig

judge = OpenAIModel(
    model="gpt-4o-mini",
    temperature=0.0,
)

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case1 = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)
test_case2 = LLMTestCase(
    input="Who built GPT models",
    actual_output="Open AI",
    retrieval_context=["Open AI built GPT models"],
)

evaluation_result = evaluate(
    test_cases=[test_case1, test_case2],
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)


In [ ]:
# Evaluation dataset
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import OpenAIModel
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

answer_relevancy_metric = AnswerRelevancyMetric()

golden = Golden(
    input="Capital of India",
    expected_output="Delhi",
    context=["New Delhi is the capital of India"],
)

dataset = EvaluationDataset()
dataset.add_golden(golden)

dataset


In [ ]:
# Creating a testcase from golden

for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,
        expected_output=golden.expected_output,
        actual_output= "Delhi",
        retrieval_context=golden.context,
    )
    
    dataset.add_test_case(test_case)
    
    evaluation_result = evaluate(
    test_cases=dataset.test_cases,
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)

In [ ]:
# Create multiple set of datasets
# Create goldens from code and push to confidentAi 


test_data = [
    {
        "input": "What is the capital of India?",
        "expected_output": "New Delhi",
        "context": ["New Delhi is the capital of India."]
    },
    {
        "input": "Who developed the Python programming language?",
        "expected_output": "Guido van Rossum",
        "context": ["Python was created by Guido van Rossum."]
    },
    {
        "input": "What is the largest planet in our solar system?",
        "expected_output": "Jupiter",
        "context": ["Jupiter is the largest planet in the solar system."]
    },
    {
        "input": "What is the boiling point of water?",
        "expected_output": "100°C at standard atmospheric pressure.",
        "context": ["Water boils at 100°C at standard atmospheric pressure."]
    },
    {
        "input": "Who built the GPT models?",
        "expected_output": "OpenAI",
        "context": ["OpenAI developed the GPT family of models."]
    }
]



In [ ]:
# Create the goldens
from deepeval.dataset import EvaluationDataset, Golden
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=data["context"],
    )
    goldens.append(golden)

new_dataset = EvaluationDataset(goldens=goldens)
new_dataset

In [12]:
new_dataset.push(alias="GoldenDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=13035583;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k]8;;\

In [ ]:
# pull the data from confidnetAI
